Automating manual retrieval and data cleaning of energy datasets by building a pipeline. 

In [1]:
import pandas as pd
import json

In [2]:
def extract_tabular_data(file_path: str):
    """Extract data from a tabular file_format, with pandas."""
    if file_path.endswith(".csv"):
        raw_data = pd.read_csv(file_path)
    elif file_path.endswith(".parquet"):
        raw_data = pd.read_parquest(file_path)
    else:
        raise Exception("Warning: Invalid file extension. Please try with .csv or .parquet!")
    return raw_data

In [3]:
def extract_json_data(file_path):
    """Extract and flatten data from a JSON file."""
    with open(file_path, 'r') as f:
        data = json.load(f)
    raw_data = pd.json_normalize(data)
    return raw_data

In [4]:
def transform_electricity_sales_data(raw_data: pd.DataFrame):
    """
    Transform electricity sales to find the total amount of electricity sold
    in the residential and transportation sectors.
    
    - Drop any records with NA values in the `price` column inplace.
    - Only keep records with a `sectorName` of "residential" or "transportation".
    - Create a `month` column using the first 4 characters of the values in `period`.
    - Create a `year` column using the last 2 characters of the values in `period`.
    - Return the transformed `DataFrame`, keeping only the columns `year`, `month`, `stateid`, `price` and `price-units`.
    """
    df = raw_data.copy()
    cleaned_data = df.dropna(subset=["price"], inplace=True)
    cleaned_data = df[df["sectorName"].isin(['residential', 'transportation'])]
    cleaned_data["month"] = cleaned_data["period"].str[:4]
    cleaned_data["year"] = cleaned_data["period"].str[-2:]
    final_df = cleaned_data[["year", "month", "stateid", "price", "price-units"]]
    return final_df

In [5]:
def load(dataframe: pd.DataFrame, file_path: str):
    """Load a DataFrame to a file in either CSV or Parquet format."""
    if file_path.endswith(".csv"):
        result = dataframe.to_csv(file_path)
    elif file_path.endswith(".parquet"):
        result = dataframe.to_parquet(file_path, compression="gzip")
    else:
        raise Exception("Warning: {filepath} is not a valid file type. Please try again!_")
    return result

In [6]:
# Test
raw_electricity_capability_df = extract_json_data("electricity_capability_nested.json")
raw_electricity_sales_df = extract_tabular_data("electricity_sales.csv")

cleaned_electricity_sales_df = transform_electricity_sales_data(raw_electricity_sales_df)

load(raw_electricity_capability_df, "loaded__electricity_capability.parquet")
load(cleaned_electricity_sales_df, "loaded__electricity_sales.csv")